# RSNA Knee 2026 — Third-place G36-Gold reproduction

This reproduces the MRI-only inference path of selected Submission **57527835**
(script **361161492**, public 0.968 / private 0.959), with all **36** final models.
The later A21GT was a separate selected submission and is not averaged here.

- [Training and inference source, environment, hardware and instructions](https://github.com/lizhihong718-alt/rsna-knee-2026-3rd-place)
- [All 36 public model weights](https://www.kaggle.com/datasets/yeeelzh/rsna-knee-2026-third-place-g36-weights)
- [Team solution](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/writeups/3rd-place-solution-rsna-knee-abnormality-detectio)

Run with **GPU T4 x2**, Internet off, and the attached competition + model dataset.
All model tensors and inference equations are preserved. The weights dataset also
contains the complete offline source archive and its licenses. The notebook wrapper
is Apache-2.0; this does not relicense the loaded package. DataTech original
contributions use CC-BY-NC-4.0; DINOv3 and Apache-2.0 upstream components retain
their licenses. No OAI data are required at inference.

The notebook computes a complete submission in a temporary private directory,
checks all 36 models and the output schema/order/range, then publishes only
`reproduction_audit.json`. No subject IDs, predictions or execution logs are
published in its outputs. To retain `submission.csv`, use a **private copy** and
set `KEEP_OUTPUTS=True`; predictions will be in `private_predictions/`.
There is no Kaggle submission API call in this notebook.

For a local output file, use `python infer.py --data ... --weights ... --out ...`
from the linked repository. Training code/configurations are available there;
exact training requires the separately authorized frozen inputs documented by
hash. This notebook reproduces inference, and does not retrain the models.


In [ ]:
# SPDX-License-Identifier: Apache-2.0 (notebook wrapper only)
from pathlib import Path
import os, sys, json, hashlib, zipfile, tempfile, subprocess, shutil, time

# Keep this False in a public notebook. A private copy may retain its predictions.
KEEP_OUTPUTS = False
SOURCE_SHA256 = "012272237f40ce84241e5364765853b2c4aa1879a6b79aeafb625d86f164e106"
DATASET_SLUG = "rsna-knee-2026-third-place-g36-weights"

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8 << 20), b""):
            h.update(block)
    return h.hexdigest()

weights = next((p for p in [Path("/kaggle/input") / DATASET_SLUG,
                          Path("/kaggle/input/datasets/yeeelzh") / DATASET_SLUG]
                if (p / "model_manifest.json").is_file()), None)
if weights is None:
    raise FileNotFoundError("Attach the public G36 weights dataset")
data = next((p for p in [Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                       Path("/kaggle/input/rsna-knee-abnormality-detection")]
             if (p / "test.csv").is_file()), None)
if data is None:
    raise FileNotFoundError("Attach the competition data under its access terms")
assert sha256(weights / "source.zip") == SOURCE_SHA256, "Source archive identity differs"
scratch = Path(tempfile.mkdtemp(prefix="g36-reproduction-"))
solution = scratch / "solution"
solution.mkdir()
with zipfile.ZipFile(weights / "source.zip") as archive:
    for item in archive.infolist():
        if not (solution / item.filename).resolve().is_relative_to(solution):
            raise ValueError("Unsafe archive member")
    archive.extractall(solution)

started = time.monotonic()
env = dict(os.environ, HF_HUB_OFFLINE="1", ENSEMBLE_RUN_DIR=str(scratch / "run"),
           ENSEMBLE_CACHE_DIR=str(scratch / "cache"), ENSEMBLE_MEMBER_DIR=str(scratch / "members"),
           ENSEMBLE_EVAL_AMP="fp16", ENSEMBLE_PROFILE="1")
with (scratch / "private_execution.log").open("w") as log:
    subprocess.run([sys.executable, str(solution / "verify.py"), "--weights", str(weights)],
                   env=env, stdout=log, stderr=subprocess.STDOUT, check=True)
    subprocess.run([sys.executable, str(solution / "infer.py"), "--data", str(data),
                    "--weights", str(weights), "--out", str(scratch / "submission.csv")],
                   env=env, stdout=log, stderr=subprocess.STDOUT, check=True)

import numpy as np
import pandas as pd
frame = pd.read_csv(scratch / "submission.csv")
test = pd.read_csv(data / "test.csv")
manifest = json.loads((weights / "model_manifest.json").read_text())
labels = json.loads((solution / "inference/members.json").read_text())["labels"]
receipt = json.loads((scratch / "submission.receipt.json").read_text())
assert list(frame.columns) == ["StudyInstanceUID", *labels]
assert frame.StudyInstanceUID.astype(str).tolist() == test.StudyInstanceUID.astype(str).tolist()
values = frame[labels].to_numpy()
assert np.isfinite(values).all() and (values >= 0).all() and (values <= 1).all()
assert receipt["model_count"] == 36 and receipt["member_ids"] == [m["id"] for m in manifest["models"]]
assert receipt["scored_fraction"] == 1 and not receipt["degraded"] and not receipt["failed_studies"]
assert not receipt["dropped_workers"] and not receipt["member_failures"]
audit = dict(status="PASS", models=36, studies=len(frame), schema_order_range="PASS",
             all_members_all_studies=True, degraded=False, weight_identity="PASS",
             source_archive_sha256=SOURCE_SHA256, elapsed_s=time.monotonic()-started,
             original_winning_submission=57527835, original_script_version=361161492,
             timing=receipt["timing"], devices=sorted({w["device"] for w in receipt["workers"].values()}))
Path("/kaggle/working/reproduction_audit.json").write_text(json.dumps(audit, indent=2))
print(json.dumps(audit, indent=2))
if KEEP_OUTPUTS:
    destination = Path("/kaggle/working/private_predictions")
    shutil.copytree(scratch, destination, dirs_exist_ok=False)
else:
    # Publish only aggregate verification; do not expose study identifiers, predictions or logs.
    shutil.rmtree(scratch)
